In [ ]:
import pandas as pd
import os

import pickle
from pathlib import Path

In [ ]:
data = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/data/2025-12-02_get_age_for_case_and_raw_control_cohorts" 

In [ ]:
# This query represents dataset "control cohort demo and vax data" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
    
def get_control_demo_table():
    
    dataset_28424625_person_sql = """
        SELECT
            person.person_id,
            person.gender_concept_id,
            p_gender_concept.concept_name as gender,
            person.birth_datetime as date_of_birth,
            person.race_concept_id,
            p_race_concept.concept_name as race,
            person.ethnicity_concept_id,
            p_ethnicity_concept.concept_name as ethnicity,
            person.sex_at_birth_concept_id,
            p_sex_at_birth_concept.concept_name as sex_at_birth,
            person.self_reported_category_concept_id,
            p_self_reported_category_concept.concept_name as self_reported_category 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
                ON person.gender_concept_id = p_gender_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
                ON person.race_concept_id = p_race_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
                ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
                ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept 
                ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
        WHERE
            person.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_ehr_data = 1 ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 
                UNION
                DISTINCT SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_lr_whole_genome_variant = 1 
                UNION
                DISTINCT SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_array_data = 1 ) 
                AND cb_search_person.person_id NOT IN (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (440029)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 )) criteria ) )"""

    dataset_28424625_person_df = pd.read_gbq(
        dataset_28424625_person_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook")

    return dataset_28424625_person_df

In [ ]:
def create_df_pkl(df, directory):

    # 2) Choose a workspace folder for persistence
    out_file = Path(directory)

    # 3) Save the entire dict in one go
    with open(out_file, 'wb') as f:
        pickle.dump(df, f)

    print(f"Saved {len(df)} DataFrames to {out_file}")

In [ ]:
##Function calls

control = get_control_demo_table()
create_df_pkl(control, f'{data}/control_cohort_demo_df.pkl')